# Patent Topic Modeling with TF‑IDF + NMF


In [3]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import NMF
import re
from pathlib import Path
import numpy as np
import itertools
import subprocess
import tempfile
import os
import pandas as pd
from sklearn.decomposition import NMF
from sklearn.metrics.pairwise import cosine_similarity

# specify which corpus version to load (used to build the file path below)
version_prefix = "v4"

## Steps 1: Load, clean, and vectorize

This cell reads the processed claims file for the selected `version_prefix`,
applies the custom stop‑word list, cleans the text, and produces a TF‑IDF
matrix in a single block.

In [4]:
# load data for the chosen prefix
base_data = Path("../../data/claims_added")
data_path = base_data / f"{version_prefix}_processed.csv"

df = pd.read_csv(data_path)

# load custom stopwords from file
stopfile = Path("custom_stopwords.txt")
if not stopfile.exists():
    raise FileNotFoundError(f"custom stopwords file not found: {stopfile}")
with open(stopfile, "r") as f:
    custom_stopwords = [line.strip() for line in f if line.strip()]

# expand multi-word entries so vectorizer tokenization doesn't warn
expanded = set()
for w in custom_stopwords:
    expanded.add(w)
    for part in w.split():
        expanded.add(part)
custom_stopwords = list(expanded)

stop_words = set(ENGLISH_STOP_WORDS).union(custom_stopwords)
stop_words = set(w.lower() for w in stop_words)

# text cleaning utility
def clean_text(s: str) -> str:
    s = s.lower()
    s = re.sub(r"[^a-z0-9_\s\-]", " ", s)         # keep alnum/_/- as token-friendly
    s = re.sub(r"\b\d{1,2}\b", " ", s)             # remove standalone 1-2 digit numbers
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_clean"] = df["claims"].map(clean_text)

# vectorize with TF‑IDF
vectorizer = TfidfVectorizer(
    stop_words=list(stop_words),
    ngram_range=(1, 2),
    min_df=10,
    max_df=0.4,
    lowercase=True,
    sublinear_tf=True,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z0-9_]{1,}\b"
)

X = vectorizer.fit_transform(df["text_clean"])
feature_names = np.array(vectorizer.get_feature_names_out())

print(f"TF‑IDF matrix shape: {X.shape}  (docs × terms)")
print(f"Vocabulary size: {len(feature_names):,}")

TF‑IDF matrix shape: (6031, 16118)  (docs × terms)
Vocabulary size: 16,118


## Step 2 : Fit Demo NMF Topic Model

In [6]:
# NMF hyperparameters
n_topics = 26

nmf = NMF(
    n_components=n_topics,
    init="nndsvda",
    random_state=42,
    max_iter=5000,
    alpha_W=0,        # regularization on W (doc-topic)
    alpha_H=0.3        # regularization on H (topic-term) forces topics to be more distinct
)

W = nmf.fit_transform(X)  # doc-topic matrix
H = nmf.components_       # topic-term matrix

print(f"W shape: {W.shape} (docs × topics)")
print(f"H shape: {H.shape} (topics × terms)")
print(f"Reconstruction error: {nmf.reconstruction_err_:.4f}")


W shape: (6031, 26) (docs × topics)
H shape: (26, 16118) (topics × terms)
Reconstruction error: 74.1607


/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 5000 reached. Increase it to improve convergence.
  warnings.warn(


## Step 3 — Inspect Topics

In [7]:
def print_top_words(H, feature_names, n_top_words=15):
    for topic_idx, topic in enumerate(H):
        top = np.argsort(topic)[::-1][:n_top_words]
        words = feature_names[top]
        weights = topic[top]
        print(f"Topic {topic_idx:02d}: " + ", ".join([f"{w}" for w in words]))

print_top_words(H, feature_names, n_top_words=20)


Topic 00: computer readable, transitory, non transitory, transitory computer, cause, readable storage, storage, storage medium, readable medium, executed, instructions executed, processors, program, code, medium instructions, executable, computer implemented, instructions cause, cause processor, implemented
Topic 01: value, according, quantization, target, threshold, number, maximum, quantized, step, calculating, values, range, error, parameter, obtain, equal, obtained, predetermined, minimum, quantizing
Topic 02: simd, instruction, single instruction, multiple data, instruction multiple, single, data simd, multiple, simd instruction, simd instructions, register, operation, simd processor, parallel, lane, result, source, loop, operand, operations
Topic 03: neural, neural network, network, layer, layers, weight, weights, network model, training, activation, input, output, quantized, quantization, layer neural, convolution, model, parameters, convolutional, loss
Topic 04: signal, signals

## Step 4: Checking number of topics, stability across seeds

In [5]:
from stability import compute_topic_stability

k_values = [16, 18, 20, 22, 24, 26, 28]
seeds = [0, 1, 2, 3, 4]

results_df = compute_topic_stability(k_values, seeds, X, method='nmf')

print("\n===== SUMMARY =====")
print(results_df.sort_values("k"))


===== k = 16 =====


/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve 

Avg Reconstruction Error: 74.7550
Omega (theta-side):      0.997
Omega (words-side):      0.998
Omega (avg):             0.998
Omega SE (repo-style):   0.0002
Avg Max Topic Cosine:    0.291
Avg Mean Topic Cosine:   0.121

===== k = 18 =====


/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve 

Avg Reconstruction Error: 74.6345
Omega (theta-side):      0.905
Omega (words-side):      0.906
Omega (avg):             0.906
Omega SE (repo-style):   0.0342
Avg Max Topic Cosine:    0.307
Avg Mean Topic Cosine:   0.112

===== k = 20 =====


...

Avg Reconstruction Error: 74.2796
Omega (theta-side):      0.914
Omega (words-side):      0.922
Omega (avg):             0.918
Omega SE (repo-style):   0.0101
Avg Max Topic Cosine:    0.313
Avg Mean Topic Cosine:   0.091

===== k = 26 =====


/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve 

Avg Reconstruction Error: 74.1643
Omega (theta-side):      0.880
Omega (words-side):      0.886
Omega (avg):             0.883
Omega SE (repo-style):   0.0098
Avg Max Topic Cosine:    0.292
Avg Mean Topic Cosine:   0.086

===== k = 28 =====


/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/sklearn/decomposition/_nmf.py:1759: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve 

Avg Reconstruction Error: 74.0580
Omega (theta-side):      0.841
Omega (words-side):      0.840
Omega (avg):             0.840
Omega SE (repo-style):   0.0108
Avg Max Topic Cosine:    0.286
Avg Mean Topic Cosine:   0.081

===== SUMMARY =====
    k  omega_theta  omega_words     omega  omega_se  max_cosine  mean_cosine  \
0  16     0.997499     0.998180  0.997840  0.000197    0.291353     0.121087   
1  18     0.905412     0.905931  0.905671  0.034161    0.306724     0.111934   
2  20     0.966444     0.967451  0.966948  0.010063    0.314624     0.104766   
3  22     0.913346     0.919176  0.916261  0.016432    0.304882     0.097223   
4  24     0.913745     0.922234  0.917990  0.010133    0.312539     0.090775   
5  26     0.879856     0.885609  0.882733  0.009805    0.292260     0.085800   
6  28     0.840908     0.839798  0.840353  0.010850    0.285757     0.080657   

   recon_error  
0    74.755049  
1    74.634472  
2    74.511999  
3    74.395025  
4    74.279636  
5    74.164318 